# 01 - Data Exploration: CAMELS-IND Dataset

This notebook explores the CAMELS-IND dataset:
- Load streamflow and meteorological data
- Examine data quality and missing values
- Visualize spatial and temporal patterns
- Identify extreme events (floods/droughts)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import CAMELSIndDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# Load data (generate synthetic if real data not available)
loader = CAMELSIndDataLoader('../data/raw')

if not loader.list_catchments():
    print('No real data found. Generating synthetic sample data...')
    loader.generate_sample_data(n_catchments=10, n_days=3650)

catchments = loader.list_catchments()
print(f'Available catchments: {len(catchments)}')
print(f'IDs: {catchments}')

In [ ]:
# Load a single catchment
data = loader.load_catchment(catchments[0])
print(f'Shape: {data.shape}')
print(f'Date range: {data["date"].min()} to {data["date"].max()}')
data.describe()

In [ ]:
# Missing value analysis
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
print('Missing values (%):')
print(missing_pct[missing_pct > 0])

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(data.isnull().T, cbar=True, cmap='viridis', ax=ax)
ax.set_title('Missing Data Pattern')
plt.tight_layout()
plt.show()

In [ ]:
# Streamflow time series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(data['date'], data['streamflow_mm_day'], linewidth=0.5)
axes[0].set_ylabel('Streamflow (mm/day)')
axes[0].set_title(f'Daily Streamflow - {catchments[0]}')

# Mark extreme events
q95 = data['streamflow_mm_day'].quantile(0.95)
q05 = data['streamflow_mm_day'].quantile(0.05)
axes[0].axhline(y=q95, color='red', linestyle='--', label=f'95th percentile ({q95:.2f})')
axes[0].axhline(y=q05, color='brown', linestyle='--', label=f'5th percentile ({q05:.2f})')
axes[0].legend()

axes[1].plot(data['date'], data['precipitation_mm'], linewidth=0.5, color='blue')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].set_xlabel('Date')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Seasonal patterns
data['month'] = pd.to_datetime(data['date']).dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data.groupby('month')['streamflow_mm_day'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Mean Monthly Streamflow')
axes[0].set_ylabel('Streamflow (mm/day)')

data.groupby('month')['precipitation_mm'].mean().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Mean Monthly Precipitation')
axes[1].set_ylabel('Precipitation (mm)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = data.select_dtypes(include=[np.number]).columns
corr = data[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()